In [92]:
from typing import Optional

import conllu
import pandas as pd
from conllu import parse_tree

In [4]:
sentences = []
with open("train_nlprepl-ud.conllu", "r", encoding="utf-8") as f:
    for sentence in conllu.parse_incr(f):
        sentences.append(sentence)

In [178]:

# Load CSV and set 'lemma' as string index
morph_dict = pd.read_csv(
    "dictionary.v3.csv",
    dtype={"lemma": "string"}  # ensure lemma is treated as string
)
morph_dict.set_index("lemma", inplace=True)

def has_lemma(lemma: str):
    return lemma in morph_dict.index

poliform_tag_by_ud_tag = {
    "ppron3:sg:nom:m3:ter:akc:npraep": "ppron3:sg:nom:m1.m2.m3:ter:_:_",
    "praet:sg:m1:perf": "praet:sg:m1.m2.m3:perf",
    "praet:sg:m2:perf": "praet:sg:m1.m2.m3:perf",
    "praet:sg:m3:perf": "praet:sg:m1.m2.m3:perf",
    "praet:sg:m1:imperf": "praet:sg:m1.m2.m3:imperf",
    "praet:sg:m2:imperf": "praet:sg:m1.m2.m3:imperf",
    "praet:sg:m3:imperf": "praet:sg:m1.m2.m3:imperf",
}

def get_form(lemma:str, tag: str) -> str:
    if not has_lemma(lemma):
        return None
    row = morph_dict.loc[lemma]

    if tag in poliform_tag_by_ud_tag:
        tag = poliform_tag_by_ud_tag[tag]

    if tag not in row.index:
        assert False, f'Invalid tag: {tag}'

    val = row[tag]
    if pd.notna(val) and val != "":
        return val
    else:
        return None

Predicate - token that:
* is root, or
* is the HEAD of a subject token (can be checked with 'deprel' dependency relation label)

In [ ]:
from copy import deepcopy
from tqdm import tqdm

def sentence_text(sent):
    """Conlu library doesn't have 'generate raw text' function"""
    out = []
    for tok in sent:
        if not isinstance(tok["id"], int):
            continue

        out.append(tok["form"])

        misc = tok.get("misc")
        if not misc or misc.get("SpaceAfter") != "No":
            out.append(" ")

    return "".join(out).rstrip()

In [39]:
s = sentences[2896]
print(s.metadata['text'])
tree = s.to_tree()
tree.print_tree()

In [91]:
s = sentences[54104]
tree = s.to_tree()
tree.print_tree()

Kiedy tak będzie - wybiorę związek.
(deprel:root) form:wybiorę lemma:wybrać upos:VERB [5]
    (deprel:advcl) form:będzie lemma:być upos:VERB [3]
        (deprel:advmod) form:Kiedy lemma:kiedy upos:ADV [1]
        (deprel:advmod:emph) form:tak lemma:tak upos:ADV [2]
        (deprel:punct) form:- lemma:- upos:PUNCT [4]
    (deprel:nsubj) form:związek lemma:związek upos:NOUN [6]
    (deprel:punct) form:. lemma:. upos:PUNCT [7]


In [87]:
def match_predicate_with_pron_nsubj(sentence) -> Optional[conllu.Token]:
    # For now lets take only root predicates
    root = sentence.to_tree()
    token = root.token

    upos = token['upos'] # Universal Part-of-Speech
    if upos != 'VERB':
        return None

    has_subj = False
    for ch in root.children:
        ch_token = ch.token
        if ch_token['deprel'] == 'nsubj' and ch_token['upos'] == "PRON":
            has_subj = True

    if not has_subj:
        return None

    return token

def change_morph(token, source_tag, target_tag):
    xpos = token['xpos'] # language-specific part-of-speech tag
    if xpos != source_tag:
        return None

    lemma = token["lemma"]
    if not has_lemma(lemma):
        return None

    is_capitalized = token['form'][0].isupper()

    target_form = get_form(lemma, target_tag)
    if target_form is None:
        return None

    if is_capitalized:
        target_form = target_form[:1].upper() + target_form[1:]

    token['form'] = target_form
    token['xpos'] = target_tag

    return token

(deprel:root) form:robisz lemma:robić upos:VERB [5]
    (deprel:advmod) form:czemu lemma:czemu upos:ADV [2]
        (deprel:advmod:emph) form:a lemma:a upos:CCONJ [1]
    (deprel:nsubj) form:ty lemma:ty upos:PRON [3]
    (deprel:advmod:neg) form:nie lemma:nie upos:PART [4]
    (deprel:punct) form:? lemma:? upos:PUNCT [6]


In [115]:
def run_filter_transform(match_token_fn, transform_inplace_fn, limit=None):
        # INLINED 'subj' filter
    rows = []

    for ix, sentence in tqdm(enumerate(sentences)):
        if limit is not None:
            if len(rows) >= limit:
                break

        sentence = deepcopy(sentence)  # to not break original objects

        token = match_token_fn(sentence)
        if token is None:
            continue

        modified_token = transform_inplace_fn(token)
        if modified_token is None:
            continue

        correct_text = sentence.metadata['text']
        incorrect_text = sentence_text(sentence)
        rows.append((ix, correct_text, incorrect_text))

    return pd.DataFrame(rows, columns=["conllu_index", "correct", "incorrect"])

In [116]:
from functools import partial

dfs = []

for time in ["perf", "imperf"]:
    persons = ["pri", "sec", "ter"]
    for source_person in persons:
        for target_person in persons:
            if source_person == target_person:
                continue

            source_tag = f'fin:sg:{source_person}:{time}'
            target_tag = f'fin:sg:{target_person}:{time}'

            transformation = partial(change_morph, source_tag=source_tag, target_tag=target_tag)
            dfs.append(run_filter_transform(match_predicate_with_pron_nsubj, transformation))

combined = pd.concat(dfs)
combined = combined.sort_values('conllu_index')
combined.to_csv("output/person_changed.csv", index=False)

In [127]:
def preview_sentences(match_token_fn, limit=5):
    matched = 0
    for ix, sentence in tqdm(enumerate(sentences)):
        if limit is not None:
            if matched >= limit:
                break
        token = match_token_fn(sentence)
        if token is not None:
            print(f"Conllu index={ix}, sentence= {sentence.metadata['text']}")
            matched += 1

69360it [00:11, 6028.19it/s] 
69360it [00:10, 6489.16it/s] 
69360it [00:10, 6388.29it/s] 
69360it [00:10, 6313.03it/s] 
69360it [00:11, 6177.45it/s] 
69360it [00:11, 6123.96it/s] 
69360it [00:11, 6302.21it/s] 
69360it [00:10, 6535.62it/s] 
69360it [00:10, 6621.72it/s] 
69360it [00:10, 6640.69it/s] 
69360it [00:10, 6601.30it/s] 
69360it [00:10, 6589.97it/s] 


In [173]:
def match_pred_with_pron_nsubj(sentence):
    root = sentence.to_tree()
    token = root.token

    upos = token['upos'] # Universal Part-of-Speech
    if upos != 'VERB':
        return None

    has_subj = False
    for ch in root.children:
        ch_token = ch.token
        if ch_token['deprel'] == 'nsubj' and ch_token['upos'] == "PRON":
            has_subj = True

    if not has_subj:
        return None

    return token

preview_sentences(partial(match_pred_with_pron_nsubj), limit=25)

1276it [00:00, 69765.64it/s]

Conllu index=112, sentence= A oni na przemian śpiewali i śmiali się z uciechy.
Conllu index=165, sentence= Lecz oni milczeli również ogarnięci właściwą powszechności niechęcią do wyjawienia swej opinii, nie poznawszy wprzód opinii drugich.
Conllu index=197, sentence= Jeżeli, mój Boże, ona tak pyta, to znaczy, coś się w niej przesiliło, chce żyć.
Conllu index=218, sentence= I wszyscy wnet pojęli, że ten okręt oznaczał Polskę, a była tam następnie mowa o nieładzie, co wkradł się do załogi, o burzy i o napadzie korsarzy, i o tym, jak okręt zaczął tonąć.
Conllu index=221, sentence= Nie pamiętała ona już domu swego zamożnym.
Conllu index=235, sentence= Lecz przekonał on się już nieraz, że przedmioty, których komu odmówił, traciły potem dla niego całą wartość.
Conllu index=354, sentence= Pod wpływem tych niewinnych zajęć przedstawiała się ona łagodniej, lecz i bardziej mglisto.
Conllu index=469, sentence= Lecz równie srogo zwykł był on występować przy rannych raportach; areszt i chłosta dykt

In [164]:
s = sentences[1189]
s.to_tree().print_tree()
print(s.to_tree().token['xpos'])
print(s.to_tree().children[0].token['xpos'])

1276it [00:00, 70833.98it/s]

Conllu index=112, sentence= A oni na przemian śpiewali i śmiali się z uciechy.
Conllu index=165, sentence= Lecz oni milczeli również ogarnięci właściwą powszechności niechęcią do wyjawienia swej opinii, nie poznawszy wprzód opinii drugich.
Conllu index=197, sentence= Jeżeli, mój Boże, ona tak pyta, to znaczy, coś się w niej przesiliło, chce żyć.
Conllu index=218, sentence= I wszyscy wnet pojęli, że ten okręt oznaczał Polskę, a była tam następnie mowa o nieładzie, co wkradł się do załogi, o burzy i o napadzie korsarzy, i o tym, jak okręt zaczął tonąć.
Conllu index=221, sentence= Nie pamiętała ona już domu swego zamożnym.
Conllu index=235, sentence= Lecz przekonał on się już nieraz, że przedmioty, których komu odmówił, traciły potem dla niego całą wartość.
Conllu index=354, sentence= Pod wpływem tych niewinnych zajęć przedstawiała się ona łagodniej, lecz i bardziej mglisto.
Conllu index=469, sentence= Lecz równie srogo zwykł był on występować przy rannych raportach; areszt i chłosta dykt

In [ ]:
dfs = []

for time in ["perf", "imperf"]:
    gender = ["f", "m1", "m2", "m3"]
    for source_gender in gender:
        for target_gender in gender:
            if source_gender == target_gender:
                continue
            if source_gender.startswith("m") and target_gender.startswith("m"):
                continue

            source_tag = f'praet:sg:{source_gender}:{time}'
            target_tag = f'praet:sg:{target_gender}:{time}'

            transformation = partial(change_morph, source_tag=source_tag, target_tag=target_tag)
            dfs.append(run_filter_transform(match_pred_with_pron_nsubj, transformation))

combined = pd.concat(dfs)
combined = combined.sort_values('conllu_index')
combined.to_csv("output/gender_changed.csv", index=False)

20433it [00:03, 5443.76it/s]

Problem - niespojność polimorf z UD, np:

* w UD mamy 'praet:pl:m1:imperf' a w polimorf nie ma
* w UD mamy 'ppron3:sg:nom:m3:ter:akc:npraep', a w polimorf nie ma  ( 'ppron3:sg:nom:m1.m2.m3:ter:_:_' ?)  (ona/on)


Problem - niespojność polimorf z UD, np:

* w UD mamy 'praet:pl:m1:imperf' a w polimorf nie ma
* w UD mamy 'ppron3:sg:nom:m3:ter:akc:npraep', a w polimorf nie ma  ( 'ppron3:sg:nom:m1.m2.m3:ter:_:_' ?)  (ona/on)
